# MCP Concepts 01: Building an MCP Server with FastMCP (Tools, Resources, Prompts)

## Problem card

- **What is MCP?** The Model Context Protocol is a standard way for an LLM
  application (the **client**) to talk to a separate process (the
  **server**) that exposes capabilities -- tools it can call, data it can
  read, and prompt templates it can reuse -- over a well-defined transport,
  instead of every app inventing its own bespoke plugin format.
- **Why a *protocol* instead of just calling a Python function?** A real MCP
  server is its own process (possibly written in a different language,
  possibly running on a different machine) that any MCP-speaking client can
  connect to -- the same server can serve a notebook, a production agent, and
  a totally different vendor's app, without any of them sharing code.
- **This notebook's scope:** build one real MCP server with
  [FastMCP](https://gofastmcp.com), run it as its own process, and connect to
  it with the raw `mcp` Python SDK client -- no LangGraph yet. That comes in
  notebook 02, once the protocol mechanics here are second nature.
- **Success criteria:** every call below is a real round trip to a real
  separate process (`servers/knowledge_ops_server.py`), not an in-process
  function call standing in for MCP.

## The architecture

```mermaid
flowchart LR
    subgraph Client Process
        C[MCP Client\ne.g. this notebook,\nor a LangGraph agent]
    end
    subgraph Server Process
        S[MCP Server\ne.g. knowledge_ops_server.py]
        T[Tools]
        R[Resources]
        P[Prompts]
        S --> T
        S --> R
        S --> P
    end
    C <-->|"transport: stdio or HTTP\n(JSON-RPC 2.0 messages)"| S

    style C fill:#a8dadc,stroke:#333
    style S fill:#f9c74f,stroke:#333
```

The client and server are **two separate processes** talking JSON-RPC over a
transport. This notebook uses **stdio**: the client spawns the server as a
subprocess and talks to it over its stdin/stdout pipes -- perfect for a local
tool the client fully controls. Notebook 04 uses the other common transport,
**Streamable HTTP**, for a server the client does *not* control (a public,
already-running server on the internet).

In [1]:
import os
import sys
import warnings
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
# Jupyter/IPython kernels already run inside an event loop and support
# top-level `await` directly in a cell -- no asyncio.run() or nest_asyncio
# needed (and nest_asyncio actually breaks the mcp SDK's anyio-based event
# loop detection, so deliberately not used here).

load_dotenv(find_dotenv(usecwd=True))

sys.path.insert(0, os.getcwd())
import shared

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_SCRIPT = os.path.join(os.getcwd(), "servers", "knowledge_ops_server.py")
print(f"Server script: {SERVER_SCRIPT}")
print(f"Exists: {os.path.exists(SERVER_SCRIPT)}")

Server script: /Users/utsabchakraborty/Documents/Edureka_Full_Course/Live_Class_Codes/Coding_Agent_Enabled_Demo/teaching/langgraph_basics/mcp_concepts/servers/knowledge_ops_server.py
Exists: True


## The three MCP primitives, at a glance

| Primitive | Who decides to use it | Purpose | This server's examples |
|---|---|---|---|
| **Tool** | The **model** (LLM decides to call it, based on the user's question) | Perform an action / fetch dynamic data | `search_runbooks`, `get_service_status` |
| **Resource** | The **client/user** (application code reads it directly, no LLM decision needed) | Expose addressable, mostly-static data by URI | `resource://team/oncall-contacts` (static), `runbook://{service}/latest` (templated) |
| **Prompt** | The **client/user** (selects a named, parameterized template) | Reusable message templates, not raw data | `incident_summary_prompt` |

This distinction matters: a tool costs an extra LLM decision (and can be
called wrong); a resource is fetched deterministically by URI whenever the
client's code decides it needs it, with no LLM in the loop at all.

In [2]:
async def connect_and_inspect():
    params = StdioServerParameters(command="python3", args=[SERVER_SCRIPT])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            # 1. initialize -- capability negotiation. Client and server agree
            #    on protocol version and what each side supports before anything
            #    else can happen.
            init_result = await session.initialize()
            print("=== initialize() ===")
            print(f"Server: {init_result.serverInfo.name} v{init_result.serverInfo.version}")
            print(f"Protocol version: {init_result.protocolVersion}")

            # 2. list_tools -- ask the server what actions it exposes.
            tools = await session.list_tools()
            print("\n=== list_tools() ===")
            for t in tools.tools:
                print(f"- {t.name}: {t.description}")

            # 3. list_resources + list_resource_templates -- static vs. dynamic data.
            resources = await session.list_resources()
            templates = await session.list_resource_templates()
            print("\n=== list_resources() ===")
            for r in resources.resources:
                print(f"- {r.uri}")
            print("\n=== list_resource_templates() ===")
            for t in templates.resourceTemplates:
                print(f"- {t.uriTemplate}")

            # 4. list_prompts -- named, reusable message templates.
            prompts = await session.list_prompts()
            print("\n=== list_prompts() ===")
            for p in prompts.prompts:
                print(f"- {p.name}: {p.description}")

await connect_and_inspect()

=== initialize() ===
Server: KnowledgeOps v3.4.6
Protocol version: 2025-11-25

=== list_tools() ===
- search_runbooks: Search runbook titles for services matching a keyword (e.g. 'payment', 'login').
- get_service_status: Get the current status, error rate, and last deploy time for a service.
- check_status: Check status by id. (Deliberately vague -- see orders_server.py's module
docstring; used only in notebook 03's tool-naming-collision demo, not in
notebooks 01/02.)

=== list_resources() ===
- resource://team/oncall-contacts

=== list_resource_templates() ===
- runbook://{service}/latest

=== list_prompts() ===
- incident_summary_prompt: Generates a ready-to-send prompt asking for an incident summary for one service.


## Calling a tool (model-decided action)

A real `call_tool` round trip: the client sends a JSON-RPC request naming
the tool and its arguments, the server process runs the actual Python
function, and the result comes back over the same pipe.

In [3]:
async def call_tools_demo():
    params = StdioServerParameters(command="python3", args=[SERVER_SCRIPT])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            result = await session.call_tool("get_service_status", {"service": "checkout"})
            print("get_service_status(checkout) ->", result.content[0].text)

            result = await session.call_tool("search_runbooks", {"query": "pay"})
            print("search_runbooks(pay) ->", result.content[0].text)

await call_tools_demo()

get_service_status(checkout) -> {"status":"degraded","error_rate_pct":18.4,"last_deploy_minutes_ago":40}
search_runbooks(pay) -> ["payments"]


## Reading resources (client-decided data, static and dynamic)

No LLM call happens here at all -- `read_resource` is a direct, deterministic
fetch by URI. The second example uses the `runbook://{service}/latest`
**template**: the client fills in `{service}` itself when constructing the
URI, the server's decorator extracts it automatically.

In [4]:
async def read_resources_demo():
    params = StdioServerParameters(command="python3", args=[SERVER_SCRIPT])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            static = await session.read_resource("resource://team/oncall-contacts")
            print("=== static resource: oncall-contacts ===")
            print(shared.pretty(static.contents[0].text))

            dynamic = await session.read_resource("runbook://payments/latest")
            print("\n=== templated resource: runbook://payments/latest ===")
            print(dynamic.contents[0].text)

await read_resources_demo()

=== static resource: oncall-contacts ===
"{\"checkout\": \"Priya K. (#checkout-oncall, priya.k@example.com)\", \"payments\": \"Sam T. (#payments-oncall, sam.t@example.com)\", \"search\": \"Jordan L. (#search-oncall, jordan.l@example.com)\", \"inventory\": \"Ravi N. (#inventory-oncall, ravi.n@example.com)\", \"auth\": \"Dana W. (#auth-oncall, dana.w@example.com)\"}"

=== templated resource: runbook://payments/latest ===
# Payments Service Runbook
1. Check card-issuer timeout rate -- a spike usually means an upstream
   acquirer bank issue, not a bug in our code.
2. Check acquirer-bank status pages before escalating internally.


## Getting a prompt (client-selected template)

`get_prompt` returns a filled-in message, ready to send to an LLM -- the
*client* chose which named prompt to use and supplied its arguments; no tool
-selection decision by a model was involved.

In [5]:
async def get_prompt_demo():
    params = StdioServerParameters(command="python3", args=[SERVER_SCRIPT])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            prompt = await session.get_prompt(
                "incident_summary_prompt",
                {"service": "search", "symptom": "queries are slow, no errors"},
            )
            print("Filled prompt message:")
            print(prompt.messages[0].content.text)

await get_prompt_demo()

Filled prompt message:
Write a concise incident summary for the 'search' service. Reported symptom: 'queries are slow, no errors'. Include: likely root cause, recommended next diagnostic step, and who to page.


## The same server, over a different transport (HTTP)

Same server file, same tools/resources/prompts -- launched with
`transport="http"` instead of the default stdio. Unlike stdio (where the
client spawns the server on demand), an HTTP server must already be running
before a client connects, so we start it explicitly and stop it when done.

In [6]:
server_proc = shared.start_http_server("servers/knowledge_ops_server.py", port=8931)
print("HTTP server started.")

HTTP server started.


In [7]:
from mcp.client.streamable_http import streamablehttp_client

async def call_over_http():
    async with streamablehttp_client("http://127.0.0.1:8931/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Tools over HTTP:", [t.name for t in tools.tools])
            result = await session.call_tool("get_service_status", {"service": "auth"})
            print("get_service_status(auth) over HTTP ->", result.content[0].text)

await call_over_http()

Tools over HTTP: ['search_runbooks', 'get_service_status', 'check_status']
get_service_status(auth) over HTTP -> {"status":"degraded","error_rate_pct":9.7,"last_deploy_minutes_ago":120}


In [8]:
shared.stop_server(server_proc)
print("HTTP server stopped.")

HTTP server stopped.


## stdio vs. HTTP: when to use which

| | stdio | Streamable HTTP |
|---|---|---|
| Who starts the server | The client spawns it as a subprocess | Someone else already runs it as a service |
| Typical use | Local tools you fully control (a local DB, local files, a script) | Remote/shared servers, servers behind auth, public servers |
| Lifecycle | Tied to the client process; dies when the client disconnects | Independent of any one client; many clients can connect |
| Network exposure | None -- pure local pipe | Real network surface -- needs the same security thinking as any HTTP service |

Notebook 04 connects to a real **public** HTTP MCP server (one we don't run
or control at all) and discusses the trust implications that come with that.

## Explain like I'm 12

Think of an MCP server like a vending machine with a sign taped to the front
that lists exactly what it sells (`list_tools`), what you can look at
through the glass without buying anything (`list_resources`), and a stack of
pre-written order forms you can grab and fill in (`list_prompts`). You (the
client) can look at the sign, grab a form, or press a button to buy
something -- and the vending machine (the server) does the actual work
inside, which you never have to see. The protocol is just the agreed rules
for "here's how you ask the machine what it has, and here's how you press a
button" -- so *any* vending machine that follows those rules works with
*any* person who knows how to use one, even if they've never seen that exact
machine before.

## Checkpoint questions

1. **Q: Why does `read_resource` never involve an LLM call, while `call_tool`
   usually does?**
   A: A resource is fetched by the client's own code whenever it decides it
   needs that data, by a known URI -- no decision-making is required. A tool
   is meant to be selected and invoked *by the model*, based on interpreting
   the user's request, which is exactly the kind of judgment call an LLM
   call is for.

2. **Q: What's the difference between `resource://team/oncall-contacts` and
   `runbook://{service}/latest`?**
   A: The first is a static resource -- one fixed URI, always the same data.
   The second is a resource *template* -- the `{service}` placeholder is
   filled in by the caller (e.g. `runbook://payments/latest`), so one
   decorated function serves many different concrete resources.

3. **Q: Why does the HTTP version of this notebook need `shared.start_http_server`/
   `stop_server`, but the stdio version doesn't manage a process at all?**
   A: `stdio_client(params)` spawns and tears down the server subprocess
   automatically as part of its `async with` block. An HTTP server is a
   long-running service that must already be listening before any client
   connects, and keeps running independently after a client disconnects, so
   its lifecycle has to be managed explicitly.

4. **Q: If this server's `check_status` tool were called with an unknown
   service name, what would happen?**
   A: The Python function raises a `ValueError`, which FastMCP turns into an
   MCP tool-error response sent back to the client -- the client gets a
   structured error, not a crash, and (in notebook 02+) the LLM sees that
   error and can decide how to react to it.